# Pilot Study: Sizing the Source-Reconstruction Method-Comparison Experiment

Comparing 6 source-reconstruction methods (`wCSET`, `CSET`, `MCE`, `sLORETA`, `dSPM`,
`MNE`) across all 473 regions of `sub-simf10`'s atlas, with many repetitions each, is too
expensive. This notebook runs a **pilot** on a stratified subset of regions, fits a
mixed-effects model per outcome measure to extract variance components, and simulates
power/precision curves across a grid of candidate `(k regions, r repetitions)` to
recommend the smallest design that still has adequate power -- backed by data, not a
guess.

**Design**: repeated-measures / mixed-effects, one simulated ground truth per
`(region, repetition)` reconstructed by all 6 methods (paired comparison, block =
region x repetition). Model per outcome: `outcome ~ method + (1|region) +
(0+method|region, homoscedastic) + (1|region:repetition)`, with the `method:region`
variance component -- does relative method performance depend on brain location --
the effect of primary scientific interest.

**Companion scripts** (only the two expensive data-generation steps -- everything else
runs here):
- `pilot_01_generate.py` -- simulates the pilot dataset (~4 min)
- `pilot_02_solve.py` -- solves all 6 methods x 208 runs, checkpointed (~48 min)

Region selection, checkpoint aggregation, variance-component fitting, Monte Carlo
power simulation, and plotting are `mindmap` library functions
(`mindmap.analysis.power`, `mindmap.core.power`, `mindmap.viz.power`) called directly
below -- this notebook is now the only place that runs them (the standalone
`pilot_03_analyze.py` mirror was retired as redundant once both called the same
functions), so the next pilot (a noise-level sweep, a connectome-degradation sweep,
...) can reuse this same machinery on its own conditions, either from its own notebook
or its own script.


## 🧪 - Region metadata

`depth`/`orientation`/`lobe` region metadata doesn't already exist in the codebase, so we
derive it here:
- **depth**: per-dipole fixed-orientation leadfield column norm (higher = more
  superficial/EEG-visible), averaged per region.
- **orientation** (radiality): `|cos(dipole normal, local radial direction from brain
  centroid)|` -- 1 = gyral/radial, 0 = sulcal/tangential -- averaged per region.
- **lobe**: parsed from the Desikan-Killiany region name (`ctx-{hemi}-{DKname}_{n}`) via
  the standard DK lobar grouping.

Note: the atlas has 473 regions, but `sub-simf10`'s forward solution is a **surface-only**
source space -- only 446 of those regions have any dipoles (27 subcortical/thalamic/
brainstem regions can't be used as simulation sources here). The existing
`select_uniform_regions` helper already restricts to `cortex_only=True` for this reason;
we follow the same restriction.


In [ ]:
import numpy as np
import pandas as pd

import mindmap as mm

dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

fwd = dm.load(sub, "fwd", desc="inv")
parc = dm.load(sub, "dseg", space="fs", segmentation="multiconn2022", scale="4")
ras2tkras = dm.load(sub, "ras2tkras")

region_meta = mm.analysis.power.compute_region_geometry(fwd, parc, ras2tkras)

print(f"{len(region_meta)} candidate cortical regions (of 473 atlas regions; "
      f"{473 - len(region_meta)} subcortical/thalamic/brainstem regions excluded -- no dipoles).")
region_meta["lobe"].value_counts()


## 🧪 - Stratified pilot region selection

Quantile-bin `depth` and `orientation` into tertiles (9 cells), allocate ~25 regions
proportionally across cells (so both "easy" superficial/radial and "hard" deep/tangential
regions are represented), then run farthest-point sampling *within* each cell over
standardized `(depth, orientation)` + a small-weight lobe code, so picks spread across
lobes too. A top-up pass guarantees every lobe present in the candidate pool gets >=1
region (insula only has 14/446 candidates and can lose out on every depth x orientation
cell it competes in).


In [ ]:
N_TARGET = 25
SEED = 0

selection = mm.analysis.power.stratified_region_selection(region_meta, N_TARGET, seed=SEED)

selection.to_csv("pilot_region_selection.tsv", sep="\t", index=False)
print(f"Selected {len(selection)} regions.")
print(selection["lobe"].value_counts())


## 🧬 - Running the pilot (already executed separately)

`pilot_01_generate.py` simulates 26 regions x 8 repetitions (fresh stochastic noise draw
each time, `snr_target=10`, same head model / epoch / propagation config as
`simulation_spike.ipynb`) -- ~4 minutes, `n_jobs=8`.

`pilot_02_solve.py` then solves all 6 methods on each of the 208 runs. Each run's
`phase_data` combines the `hrp`/`pp` two-timepoint columns (for localization error and
the two-timepoint PR curves) with 50 evenly-spaced columns across **gold's own active
signal duration** in one solve call per method (not the whole `signal_window` -- that
was tried first and wasted most of the 50 points on the flat pre/post-spike period where
`gold` is exactly zero, giving undefined/NaN spatial correlation; confirmed by inspecting
a smoke-test run's curve, which had only ~17/50 non-NaN points before the fix). `CSET`
and `MCE` don't scale trivially with timepoint count (~17s/~14s for 50 points vs. a
fraction of a second for the MNE-family solvers), so the realistic per-run cost is ~42s,
not the ~13s a 2-point-only solve would suggest -- this actually took ~48 min wall-clock
at `n_jobs=8` (vs. an ~18 min estimate), likely BLAS-thread oversubscription across the 8
worker processes. Checkpointed per run (`pilot_metrics/run-{run:03d}.json` +
`_curves.npz`) so a crash only loses runs not yet started.


## 📊 - Aggregate outcome measures

Three outcome measures per `(region, repetition, method)`: peak localization error at
`hrp` (half-rising-phase), average precision (AP) at `hrp`/`pp`, and the ~50-point
spatial-correlation-to-gold curve reduced to a scalar AUC-over-time summary (the raw
50-point curve is also kept, per run, in `pilot_metrics/run-{run:03d}_curves.npz` --
for a secondary cluster-based permutation test not run here -- but is not stitched into
one combined array by this step).


In [ ]:
import json
from pathlib import Path

METRICS_DIR = Path("pilot_metrics")
OUTCOMES = ["localization_error_hrp", "ap_hrp", "ap_pp", "correlation_auc"]
METHODS = ["wcset", "cset", "mce", "sloreta", "dspm", "mne"]

df_index = pd.read_csv(dm.roots["sim"] / "sub-simf10-pilot" / "simulation_index.tsv", sep="\t")

df = mm.analysis.power.load_pilot_checkpoints(df_index, METRICS_DIR, index_cols=("region", "repetition"))
print(f"Aggregated {len(df)} (region,repetition,method) rows "
      f"({df['run'].nunique()} runs x {df['method'].nunique()} methods).")
df.to_csv("pilot_outcomes.tsv", sep="\t", index=False)
df.head()


## 📊 - Fit mixed-effects models, extract variance components

`re_formula="~1"` (region random intercept) + two named variance components --
`method_region` (`0+C(method)`: one shared variance across the 6 per-region method
deviations -- `sigma2(method x region)`, the effect of primary scientific interest) and
`rep` (`0+C(repetition)`: repetition-within-region) -- instead of a full unstructured
`re_formula="~C(method)"` random-slope covariance (21 covariance parameters, confirmed
singular/non-convergent on a toy fit at this pilot's scale: 208 rows, 26 groups). This
variance-components form (4 variance parameters total) is well-identified and actually
fittable.


In [ ]:
variance_components = {}

for outcome in OUTCOMES:
    vc = mm.analysis.power.fit_variance_components(
        df, outcome,
        group_col="region", fixed_col="method",
        vc_formula={"method_region": "0 + C(method)", "rep": "0 + C(repetition)"},
        fixed_levels=METHODS,
    )
    variance_components[outcome] = {
        "sigma2_region": vc["sigma2_region"],
        "sigma2_method_region": vc["sigma2_method_region"],
        "sigma2_rep": vc["sigma2_rep"],
        "sigma2_resid": vc["sigma2_resid"],
        "method_means": vc["condition_means"],
    }

with open("pilot_variance_components.json", "w") as f:
    json.dump(variance_components, f, indent=2)

pd.DataFrame(variance_components).T[["sigma2_region", "sigma2_method_region", "sigma2_rep", "sigma2_resid"]]


## 🎯 - Simulate power/precision curves across a `(k, r)` grid

Classical two-way (region x method) split-plot ANOVA, region random, method fixed,
region:method a random interaction, `r` sub-replicates per cell:

```
SS_region      = r*M * sum_i (region_mean_i - grand_mean)^2         df = k-1
SS_method      = k*r * sum_l (method_mean_l - grand_mean)^2         df = M-1
SS_interaction = r   * sum_{i,l} (cell_mean - region_mean_i
                                  - method_mean_l + grand_mean)^2   df = (k-1)(M-1)
SS_residual    =       sum_{i,j,l} (y - cell_mean)^2                df = k*M*(r-1)
```

`F(method) = MS_method/MS_interaction` (region:method interaction is the correct error
term for the fixed method effect -- standard repeated-measures-ANOVA logic, region =
"subject"); `F(interaction) = MS_interaction/MS_residual`. Simulating cell means directly
(rather than every raw observation + a full `MixedLM` refit per iteration) is exact for
this balanced design and orders of magnitude faster -- what makes a dense `(k,r)` grid
tractable.


In [ ]:
N_SIM = 2000
ALPHA = 0.05
M = len(METHODS)
k_grid = [10, 15, 20, 26, 35, 50, 75, 100]
r_grid = [3, 5, 8, 10, 15, 20]

power_results = {o: {"condition": np.zeros((len(k_grid), len(r_grid))), "interaction": np.zeros((len(k_grid), len(r_grid)))} for o in OUTCOMES}
precision_results = {o: np.zeros(len(r_grid)) for o in OUTCOMES}

for oi, outcome in enumerate(OUTCOMES):
    vc = variance_components[outcome]
    for i, k in enumerate(k_grid):
        for j, r in enumerate(r_grid):
            seed = oi * 100_000 + i * 1000 + j
            pm, pi = mm.core.simulate_splitplot_power(
                condition_means=vc["method_means"],
                sigma2_group=vc["sigma2_region"],
                sigma2_interaction=vc["sigma2_method_region"],
                sigma2_rep=vc["sigma2_rep"],
                sigma2_resid=vc["sigma2_resid"],
                k=k, r=r, m=M, n_sim=N_SIM, alpha=ALPHA, seed=seed,
            )
            power_results[outcome]["condition"][i, j] = pm
            power_results[outcome]["interaction"][i, j] = pi
    for j, r in enumerate(r_grid):
        precision_results[outcome][j] = mm.core.precision_of_mean(vc["sigma2_rep"], vc["sigma2_resid"], r, M)

np.savez("pilot_power_results.npz", k_grid=k_grid, r_grid=r_grid,
         **{f"{o}_method": power_results[o]["condition"] for o in OUTCOMES},
         **{f"{o}_interaction": power_results[o]["interaction"] for o in OUTCOMES},
         **{f"{o}_precision": precision_results[o] for o in OUTCOMES})
print("Power grid computed for all outcomes.")


In [ ]:
fig = mm.viz.plot_power_precision_curves(power_results, precision_results, k_grid, r_grid, OUTCOMES)
fig.savefig("pilot_power_curves.png", dpi=130)
fig


## ✅ - Recommendation

**Only one test is close to the power threshold**: the `method x region` interaction
test on `localization_error_hrp`, which dips to **0.72 at the smallest grid point**
(k=10, r=3) but reaches 0.86 by k=15 and >=0.96 at r=5 for any k. Every other
outcome/test (`ap_hrp`, `ap_pp`, `correlation_auc`, and the `method` main effect for
every outcome) is already saturated near 1.0 across the *entire* grid -- the six
methods' average performance differs by so much that the main effect isn't a useful
test for sizing this study at all; the real constraint is detecting *where* relative
method ranking changes with region.

**Recommendation: k=20 regions, r=5 repetitions (100 total sims/method)** --
comfortably past the plateau for every outcome/test (interaction power >=0.99 for the
hardest outcome, ~1.0 for the rest) with a safety margin above the bare 0.8 threshold
(variance components are themselves point estimates from a single pilot and carry their
own sampling uncertainty), landing at less than half the pilot's own budget (100 vs.
208 sims) and a small fraction of a naive full-473-region sweep.

**Caveats / how to apply**:
- Select the `k` regions via the *same* depth x orientation x lobe stratification
  validated here, not simple random sampling -- the variance components were estimated
  over that stratified coverage, and a purely random subsample risks under-representing
  the anatomical extremes that likely drive `sigma2_region`/`sigma2_method_region`.
- The pilot's own dataset (k=26, r=8, already collected) already clears every
  plateau with a comfortable margin -- it can likely serve directly as the main study's
  data (or a random 5-of-8 repetitions per region subset of it), with no further
  simulation required, if independence from the pilot isn't a hard requirement.
- These variance components come from a single pilot subject (`sub-simf10`) and a single
  pilot run; if the main study is meant to generalize across subjects, this sizing
  analysis should ideally be repeated on at least one more subject before being treated
  as final.
